# 🦅 Business Entity Resolution — Kaggle GPU Runbook

Optimized pipeline for **Kaggle Notebooks** with **30 Free GPU Hours/Week** (NVIDIA T4 / P100).

### Kaggle Environment Setup:
1. In the right-hand panel under **Notebook settings**:
   - **Accelerator:** Select **GPU T4 x2** (or GPU T4).
   - **Internet:** Toggle **Internet ON**.
2. Add the competition dataset under **+ Add Input** (upload the dataset archive), or let the notebook auto-detect it under `/kaggle/input`.

In [ ]:
# 1. Clone GitHub Repository & Navigate into Student Resource
import os
if not os.path.exists('/kaggle/working/amazon-entity-resolution'):
    !git clone https://github.com/Luciferxy/amazon-entity-resolution.git /kaggle/working/amazon-entity-resolution

%cd /kaggle/working/amazon-entity-resolution/student_resource
print('Working directory:', os.getcwd())

In [ ]:
# 2. Check GPU & Hardware
import torch
print(f'PyTorch Version: {torch.__version__}')
cuda_ok = torch.cuda.is_available()
print(f'CUDA Available: {cuda_ok}')
if cuda_ok:
    for i in range(torch.cuda.device_count()):
        dev = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {dev} | VRAM: {vram:.1f} GB')
else:
    print('WARNING: GPU accelerator not active. In Kaggle right panel, set Accelerator to GPU T4 x2.')

In [ ]:
# 3. Install Required Packages
%pip install -q pandas pyarrow numpy scipy scikit-learn lightgbm rapidfuzz anyascii sparse_dot_topn lightning
import pandas, pyarrow, lightgbm, rapidfuzz, anyascii, sparse_dot_topn, lightning
print('All dependencies loaded successfully!')

In [ ]:
# 4. Detect Dataset Location
from pathlib import Path
from config import DATASET
print(f'Detected Dataset Root: {DATASET}')
assert (DATASET / 'train/train_source1.tsv').exists(), f'train_source1.tsv not found at {DATASET}! Click "+ Add Input" to attach the dataset.'

In [ ]:
# 5. Configure GPU Environment Variables
os.environ['ER_BACKEND'] = 'gpu' if torch.cuda.is_available() else 'cpu'
os.environ['ER_GPU_MODE'] = 'exact'
os.environ['ER_K'] = '5'
print(f'Configured: ER_BACKEND={os.environ["ER_BACKEND"]} | ER_GPU_MODE={os.environ["ER_GPU_MODE"]} | ER_K={os.environ["ER_K"]}')

In [ ]:
# 6. Preprocess Raw Data into Optimized Parquets under work/
!python src/prep.py train
!python src/prep.py test

In [ ]:
# 7. Train Model on 100,000 S1 Entities (~7-8 min on T4 GPU)
TRAIN_SAMPLE = 100_000
!python src/train_lightning.py {TRAIN_SAMPLE}

In [ ]:
# 8. Run Full Test Set Inference (Produces matching_results.tsv and candidate_pairs.tsv)
!python src/run_test.py

In [ ]:
# 9. Validate Submission Format
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir {str(DATASET / 'test')}

In [ ]:
# 10. Copy Final Submission Files to /kaggle/working/ for 1-Click Download
!cp output/matching_results.tsv /kaggle/working/matching_results.tsv
!cp output/candidate_pairs.tsv /kaggle/working/candidate_pairs.tsv
!zip -j /kaggle/working/submission_files.zip output/matching_results.tsv output/candidate_pairs.tsv
print('Files copied to /kaggle/working/! Check the "Output" tab on the right panel to download.')
!ls -lh /kaggle/working/*.tsv /kaggle/working/*.zip